# 01 Data Preparation - S&P 500 Universe

This notebook builds the raw dataset used by the rest of the project.

It downloads the current S&P 500 constituent list, maps tickers to Yahoo Finance format, downloads daily market data, creates the adjusted-close price matrix, builds the daily returns matrix, and saves benchmark returns for `^GSPC`.

Important limitation: the S&P 500 universe is based on the current constituent list, not point-in-time historical membership.

## Outputs Prepared

This step creates:

- current S&P 500 constituent metadata
- Yahoo-compatible ticker mappings, such as `BRK.B` to `BRK-B`
- adjusted close, close, and volume matrices
- daily stock returns
- S&P 500 benchmark returns
- data quality and sanity-check reports

In [ ]:
from io import StringIO
from pathlib import Path
from time import sleep

import numpy as np
import pandas as pd
import requests
import yfinance as yf

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

## 1. Configuration

Set the data source, benchmark ticker, date range, basic data-quality thresholds, and Yahoo Finance batch-download settings.

In [ ]:
SP500_WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
BENCHMARK_TICKER = "^GSPC"

START_DATE = "2019-01-01"
END_DATE = None  # None means use the latest available Yahoo Finance date.

# Basic data-quality thresholds.
MIN_AVG_DOLLAR_VOLUME = 5_000_000
MIN_OBSERVATIONS_FOR_ANALYSIS = 252  # Keep newer stocks, but require about 1 trading year of returns.
CLUSTERING_MIN_OVERLAP_DAYS = 252  # Pairwise correlation in notebook 02 should use at least this overlap.

# Use batch downloads to reduce timeout, cache-lock, and rate-limit issues.
YF_BATCH_SIZE = 50
YF_BATCH_SLEEP_SECONDS = 1

# ----------------------------------------------------------------------
# Project paths
# ----------------------------------------------------------------------
def find_project_root() -> Path:
    """Find the repository root from either a script run or notebook run."""
    candidates = []
    try:
        candidates.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    candidates.append(Path.cwd())

    root_markers = ("notebooks", "scripts", "data", "outputs")
    for start in dict.fromkeys(candidates):
        for candidate in [start, *start.parents]:
            if (candidate / ".git").exists() or all((candidate / marker).exists() for marker in root_markers):
                return candidate

    return Path.cwd()


PROJECT_ROOT = find_project_root()

OUTPUT_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Load S&P 500 Universe

Load the current S&P 500 constituent table from Wikipedia and convert ticker symbols to Yahoo Finance format.

In [ ]:
def to_yahoo_ticker(symbol: str) -> str:
    """Convert S&P 500 symbols to Yahoo Finance ticker format."""
    return symbol.replace(".", "-")


def get_sp500_universe(url: str = SP500_WIKI_URL) -> pd.DataFrame:
    """Load current S&P 500 constituents and metadata from Wikipedia."""
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; stock-selection-research/1.0)"},
        timeout=30,
    )
    response.raise_for_status()
    tables = pd.read_html(StringIO(response.text))
    universe = tables[0].copy()
    universe = universe.rename(
        columns={
            "Symbol": "symbol",
            "Security": "company_name",
            "GICS Sector": "sector",
            "GICS Sub-Industry": "sub_industry",
            "Headquarters Location": "headquarters",
            "Date added": "date_added",
            "CIK": "cik",
            "Founded": "founded",
        }
    )
    universe["symbol"] = universe["symbol"].astype(str).str.strip()
    universe["yahoo_ticker"] = universe["symbol"].map(to_yahoo_ticker)
    universe["date_added"] = pd.to_datetime(universe["date_added"], errors="coerce")
    universe = universe.sort_values("symbol").reset_index(drop=True)
    return universe


sp500_universe = get_sp500_universe()
STOCK_TICKERS = sp500_universe["yahoo_ticker"].tolist()
ALL_TICKERS = sorted(set(STOCK_TICKERS + [BENCHMARK_TICKER]))

print(f"S&P 500 universe size: {len(STOCK_TICKERS)}")
display(sp500_universe.head())
display(sp500_universe[sp500_universe["symbol"] != sp500_universe["yahoo_ticker"]])

## 3. Download Yahoo Finance Data

Download daily price and volume data with `auto_adjust=False` so that the `Adj Close` field is available explicitly.

In [ ]:
def chunked(values, size):
    for start in range(0, len(values), size):
        yield values[start:start + size]


def download_yahoo_prices(tickers, start, end=None, batch_size=50, pause_seconds=1):
    """Download daily OHLCV data from Yahoo Finance in batches."""
    frames = []
    batches = list(chunked(list(tickers), batch_size))

    for batch_no, batch in enumerate(batches, start=1):
        print(f"Downloading batch {batch_no}/{len(batches)}: {len(batch)} tickers")
        raw_batch = yf.download(
            tickers=batch,
            start=start,
            end=end,
            auto_adjust=False,
            actions=True,
            group_by="column",
            progress=False,
            threads=False,  # More stable in notebooks; avoids occasional yfinance cache locks.
        )
        if not raw_batch.empty:
            frames.append(raw_batch)
        sleep(pause_seconds)

    if not frames:
        raise ValueError("Yahoo Finance returned no data. Check tickers, dates, or internet connection.")

    raw = pd.concat(frames, axis=1).sort_index(axis=1)
    raw = raw.loc[:, ~raw.columns.duplicated()]
    return raw


def get_field(raw, field):
    """Extract a field from yfinance output for multiple tickers."""
    if isinstance(raw.columns, pd.MultiIndex):
        if field in raw.columns.get_level_values(0):
            data = raw[field].copy()
        elif field in raw.columns.get_level_values(1):
            data = raw.xs(field, axis=1, level=1).copy()
        else:
            raise KeyError(f"Field {field!r} not found in downloaded data.")
    else:
        if field not in raw.columns:
            raise KeyError(f"Field {field!r} not found in downloaded data.")
        data = raw[[field]].copy()
        data.columns = [ALL_TICKERS[0]]
    return data.sort_index()


raw_data = download_yahoo_prices(
    ALL_TICKERS,
    START_DATE,
    END_DATE,
    batch_size=YF_BATCH_SIZE,
    pause_seconds=YF_BATCH_SLEEP_SECONDS,
)

print(raw_data.shape)
display(raw_data.tail())

## 4. Create Price and Volume Matrices

In [ ]:
adj_close_all = get_field(raw_data, "Adj Close")
close_all = get_field(raw_data, "Close")
volume_all = get_field(raw_data, "Volume")

# Separate stocks and benchmark while preserving the universe metadata order.
adj_close = adj_close_all.reindex(columns=STOCK_TICKERS)
close = close_all.reindex(columns=STOCK_TICKERS)
volume = volume_all.reindex(columns=STOCK_TICKERS)
benchmark_adj_close = adj_close_all[BENCHMARK_TICKER].dropna()

print("Adjusted close shape:", adj_close.shape)
print("Benchmark rows:", len(benchmark_adj_close))
display(adj_close.tail())
display(benchmark_adj_close.tail().to_frame(BENCHMARK_TICKER))

## 5. Data Quality Report

Check data coverage, missing values, liquidity, and failed Yahoo Finance downloads. Newer stocks are not removed only because they have shorter histories.

In [ ]:
def build_data_quality_report(adj_prices, close_prices, volumes):
    first_valid = adj_prices.apply(lambda s: s.first_valid_index())
    last_valid = adj_prices.apply(lambda s: s.last_valid_index())
    valid_obs = adj_prices.notna().sum()
    valid_ratio = adj_prices.notna().mean()
    history_years = (last_valid - first_valid).dt.days / 365.25

    dollar_volume = close_prices * volumes
    avg_dollar_volume_60d = dollar_volume.tail(60).mean()
    avg_dollar_volume_all = dollar_volume.mean()

    report = pd.DataFrame({
        "yahoo_ticker": adj_prices.columns,
        "first_date": first_valid,
        "last_date": last_valid,
        "history_years": history_years,
        "valid_obs": valid_obs,
        "valid_ratio": valid_ratio,
        "avg_dollar_volume_60d": avg_dollar_volume_60d,
        "avg_dollar_volume_all": avg_dollar_volume_all,
    }).set_index("yahoo_ticker")

    report["pass_has_data"] = report["valid_obs"] > 0
    report["pass_liquidity"] = report["avg_dollar_volume_60d"] >= MIN_AVG_DOLLAR_VOLUME
    report["pass_basic"] = report[["pass_has_data", "pass_liquidity"]].all(axis=1)

    metadata = sp500_universe.set_index("yahoo_ticker")[["symbol", "company_name", "sector", "sub_industry"]]
    report = metadata.join(report, how="right")
    return report.sort_values(["pass_basic", "valid_ratio", "avg_dollar_volume_60d"], ascending=[True, True, False])


quality_report = build_data_quality_report(adj_close, close, volume)
basic_tickers = quality_report.index[quality_report["pass_basic"]].tolist()
failed_basic_tickers = quality_report.index[~quality_report["pass_basic"]].tolist()

print(f"Basic-pass tickers ({len(basic_tickers)}): {basic_tickers[:20]}{' ...' if len(basic_tickers) > 20 else ''}")
print(f"Basic-fail tickers ({len(failed_basic_tickers)}): {failed_basic_tickers[:20]}{' ...' if len(failed_basic_tickers) > 20 else ''}")
display(quality_report.head(20))

## 6. Create Returns Matrix

Compute daily percentage returns from adjusted close prices and align stock returns with the benchmark calendar.

In [ ]:
clean_adj_close = adj_close[basic_tickers].dropna(how="all")

# Forward-fill only after a ticker has started trading; do not fill pre-listing history.
clean_adj_close = clean_adj_close.ffill()

returns_matrix_raw = clean_adj_close.pct_change(fill_method=None).dropna(how="all")
benchmark_returns = benchmark_adj_close.pct_change(fill_method=None).rename("benchmark_return").dropna()

# Keep only dates shared by stock returns and benchmark returns.
common_index = returns_matrix_raw.index.intersection(benchmark_returns.index)
returns_matrix_raw = returns_matrix_raw.loc[common_index]
benchmark_returns = benchmark_returns.loc[common_index]

# Keep stocks that started after 2019 when enough data exists; Notebook 02 handles pairwise correlation with a minimum-overlap rule.
return_observations = returns_matrix_raw.notna().sum()
return_missing_ratio = returns_matrix_raw.isna().mean()
analysis_tickers = return_observations[return_observations >= MIN_OBSERVATIONS_FOR_ANALYSIS].index.tolist()
dropped_for_short_history = return_observations[return_observations < MIN_OBSERVATIONS_FOR_ANALYSIS].sort_values()

returns_matrix = returns_matrix_raw[analysis_tickers]
benchmark_returns = benchmark_returns.loc[returns_matrix.index]
clean_adj_close = clean_adj_close[analysis_tickers]

quality_report["return_missing_ratio"] = return_missing_ratio.reindex(quality_report.index)
quality_report["return_observations"] = return_observations.reindex(quality_report.index).fillna(0).astype(int)
quality_report["pass_min_observations"] = quality_report["return_observations"] >= MIN_OBSERVATIONS_FOR_ANALYSIS
quality_report["pass_analysis"] = quality_report["pass_basic"] & quality_report["pass_min_observations"]

print("Basic-pass tickers:", len(basic_tickers))
print("Analysis tickers:", len(analysis_tickers))
print("Dropped because they have fewer than minimum return observations:", len(dropped_for_short_history))
display(dropped_for_short_history.head(20).to_frame("return_observations"))
print("Returns matrix shape:", returns_matrix.shape)
print("Returns missing max:", returns_matrix.isna().mean().max())
print("Benchmark returns shape:", benchmark_returns.shape)
display(returns_matrix.tail())
display(benchmark_returns.tail().to_frame())

## 7. Quick Sanity Check

In [ ]:
summary = pd.DataFrame({
    "daily_mean": returns_matrix.mean(),
    "daily_vol": returns_matrix.std(),
    "annual_return_est": returns_matrix.mean() * 252,
    "annual_vol_est": returns_matrix.std() * np.sqrt(252),
    "min_daily_return": returns_matrix.min(),
    "max_daily_return": returns_matrix.max(),
}).sort_values("annual_return_est", ascending=False)

summary = quality_report[["symbol", "company_name", "sector", "sub_industry"]].join(summary, how="right")
display(summary.head(20))

In [ ]:
missing_after_clean = returns_matrix.isna().mean().sort_values(ascending=False)
extreme_returns = (returns_matrix.abs() > 0.30).sum().sort_values(ascending=False)

sanity_report = pd.DataFrame({
    "missing_ratio_after_clean": missing_after_clean,
    "days_abs_return_gt_30pct": extreme_returns,
})
sanity_report = quality_report[["symbol", "company_name", "sector"]].join(sanity_report, how="right")

display(sanity_report.sort_values(["missing_ratio_after_clean", "days_abs_return_gt_30pct"], ascending=False).head(20))

## 8. Save Files for Next Steps

Save the prepared data files used by the stock-selection, allocation, and backtest notebooks.

In [ ]:
sp500_universe.to_csv(OUTPUT_DIR / "sp500_universe.csv", index=False, encoding="utf-8-sig")
adj_close.to_parquet(OUTPUT_DIR / "adj_close_raw.parquet")
close.to_parquet(OUTPUT_DIR / "close_raw.parquet")
volume.to_parquet(OUTPUT_DIR / "volume_raw.parquet")
clean_adj_close.to_parquet(OUTPUT_DIR / "adj_close_clean.parquet")
returns_matrix.to_parquet(OUTPUT_DIR / "returns_matrix.parquet")
benchmark_returns.to_frame().to_parquet(OUTPUT_DIR / "benchmark_returns.parquet")
quality_report.to_csv(OUTPUT_DIR / "data_quality_report.csv", encoding="utf-8-sig")
summary.to_csv(OUTPUT_DIR / "return_summary.csv", encoding="utf-8-sig")
sanity_report.to_csv(OUTPUT_DIR / "sanity_report.csv", encoding="utf-8-sig")

print(f"Saved files to: {OUTPUT_DIR.resolve()}")

## Next Step

Notebook 02 uses `returns_matrix.parquet` to compute correlation distance, cluster stocks, and select representative stocks from each cluster.